In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from gpt import *

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

config = GPTConfig()
gpt = GPT(config).to(device)

number of parameters: 130.83M


In [22]:
next(gpt.parameters()).dtype

torch.float32

In [24]:
# initialize a GradScaler. If enabled=False scaler is a no-op
scaler = torch.cuda.amp.GradScaler(enabled=True)

/tmp/ipykernel_704653/3139212680.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


In [3]:
B, T, C = 32, 100, 768
x = torch.rand((B,T,C), device=device)

# out = gpt(x)

# out.shape

In [11]:
v, _ = torch.topk(x[:,-1,:], 5)

In [18]:
v[:, [-1]].shape

torch.Size([32, 1])

In [6]:
x[:, [-1], :].shape

torch.Size([32, 1, 768])

In [5]:
from torch.nn.functional import scaled_dot_product_attention

In [2]:
transformer = nn.Transformer(
    d_model=64, 
    nhead=4,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    activation='relu',
    batch_first=True
)

transformer = transformer.to(device)

In [3]:
src = torch.rand((32, 10, 64), device=device)
tgt = torch.rand((32, 20, 64), device=device)
out = transformer(src, tgt)

In [4]:
out.shape

torch.Size([32, 20, 64])

### karpathy

In [2]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# version 2:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei@ x # (_, T, T) @ (B, T, C) ----> (B,T,C)

In [24]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei@x

In [3]:
# version 4: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32              # B, T, C
x = torch.randn(B,T,C)

# single Head Self Attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B,T,16)
q = query(x) # (B,T,16)
wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) --> (B,T,T)


tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x) # (B,T,16)
out = wei @ v # (B,T,T) @ (B,T,16) --> (B,T,16)
# out = wei@x

out.shape

torch.Size([4, 8, 16])

In [40]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089

In [ ]:
wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) --> (B,T,T)

In [31]:
table = torch.ones((64, 64))

idx = torch.randint(1,64, (32, 10))


In [33]:
idx.shape

torch.Size([32, 10])

In [32]:

table[idx].shape

torch.Size([32, 10, 64])

In [27]:
table.shape

torch.Size([64, 64])

In [28]:
idx.shape

torch.Size([32, 10])

In [7]:
sum_context.shape

torch.Size([4, 8, 2])